# SSI: Flujo completo Issuer → Holder usando ACA-Py (Indy ledger)

**Objetivo:** Registrar un *schema* y *credential definition* en el ledger Indy, establecer una conexión DIDComm entre un *issuer* y un *holder* (ambos como wallets multitenant en ACA-Py), emitir una credencial y almacenar esa credencial en el wallet del holder.

**Requisitos previos (local):**
- VON Network (Indy) corriendo y accesible en `http://localhost:9000/genesis`.
- Un agente ACA-Py multitenant corriendo con Admin API accesible en `http://localhost:9031` y `--admin-insecure-mode` (sin autenticación para el admin) o con tokens si tu setup lo requiere.

In [1]:
# Configuración
ADMIN_URL = 'http://localhost:9031'  # Admin API del agente ACA-Py
TIMEOUT = 5
import requests, time, json
print('Admin URL:', ADMIN_URL)

Admin URL: http://localhost:9031


In [2]:
# Funciones auxiliares para llamadas HTTP y polling
import requests, time, json

def post(url, json_payload=None, headers=None):
    print('POST', url)
    r = requests.post(url, json=json_payload, headers=headers, timeout=30)
    try:
        return r.status_code, r.json()
    except Exception:
        return r.status_code, r.text

def get(url, headers=None):
    print('GET', url)
    r = requests.get(url, headers=headers, timeout=30)
    try:
        return r.status_code, r.json()
    except Exception:
        return r.status_code, r.text

def wait_for_connection(admin, conn_id, headers=None, timeout=60):
    start = time.time()
    while time.time() - start < timeout:
        sc, body = get(f"{admin}/connections/{conn_id}", headers=headers)
        if isinstance(body, dict):
            state = body.get('state') or body.get('connection', {}).get('state')
            print('state=', state)
            if state == 'active':
                return True, body
        time.sleep(1)
    return False, None


In [3]:
# Comprobar que el agent admin responde
sc, body = get(f"{ADMIN_URL}/status")
print('status code:', sc)
print(json.dumps(body, indent=2))
assert sc == 200, 'El admin API no responde en ADMIN_URL. Revisa el agente.'

GET http://localhost:9031/status
status code: 200
{
  "version": "1.2.0",
  "label": "Agent with Local Genesis",
  "conductor": {
    "in_sessions": 0,
    "out_encode": 0,
    "out_deliver": 0,
    "task_active": 1,
    "task_done": 7,
    "task_failed": 1,
    "task_pending": 0
  }
}


In [ ]:
# Crear 2 wallets multitenant: Issuer y Holder
issuer_name = 'issuer_wallet_test'
issuer_key = 'issuerkey1234567890123456789012'
holder_name = 'holder_wallet_test'
holder_key = 'holderkey1234567890123456789012'

# Crear issuer wallet
sc, issuer_resp = post(f"{ADMIN_URL}/multitenancy/wallet", json_payload={
    'wallet_name': issuer_name,
    'wallet_key': issuer_key,
    'label': issuer_name,
    'wallet_type': 'askar',
    'key_management_mode': 'managed'
})
print('issuer create:', sc)
print(json.dumps(issuer_resp, indent=2))

# Crear holder wallet
sc, holder_resp = post(f"{ADMIN_URL}/multitenancy/wallet", json_payload={
    'wallet_name': holder_name,
    'wallet_key': holder_key,
    'label': holder_name,
    'wallet_type': 'askar',
    'key_management_mode': 'managed'
})
print('holder create:', sc)
print(json.dumps(holder_resp, indent=2))

assert isinstance(issuer_resp, dict) and 'wallet_id' in issuer_resp, 'Fallo crear issuer wallet'
assert isinstance(holder_resp, dict) and 'wallet_id' in holder_resp, 'Fallo crear holder wallet'

issuer_wallet_id = issuer_resp['wallet_id']
holder_wallet_id = holder_resp['wallet_id']

print('\nissuer_wallet_id=', issuer_wallet_id)
print('holder_wallet_id=', holder_wallet_id)


In [ ]:
# Obtener token JWT para cada wallet (multitenant)
sc, tok = post(f"{ADMIN_URL}/multitenancy/wallet/{issuer_wallet_id}/token", json_payload={'wallet_key': issuer_key})
issuer_token = tok.get('token') if isinstance(tok, dict) else None
print('issuer token:', issuer_token)

sc, tok2 = post(f"{ADMIN_URL}/multitenancy/wallet/{holder_wallet_id}/token", json_payload={'wallet_key': holder_key})
holder_token = tok2.get('token') if isinstance(tok2, dict) else None
print('holder token:', holder_token)

assert issuer_token and holder_token, 'No se obtuvieron tokens. Revisa errores.'


In [ ]:
# Con los tokens, crear DID local en cada wallet y registrarlo en ledger
issuer_headers = {'Content-Type': 'application/json', 'Authorization': f'Bearer {issuer_token}'}
holder_headers = {'Content-Type': 'application/json', 'Authorization': f'Bearer {holder_token}'}

# Crear DID local issuer
sc, did_info = post(f"{ADMIN_URL}/wallet/did/create", json_payload={'method': 'sov', 'options': {'key_type': 'ed25519'}}, headers=issuer_headers)
print('issuer did create', sc, did_info)
issuer_did = did_info.get('result', {}).get('did') if isinstance(did_info, dict) else None
issuer_verkey = did_info.get('result', {}).get('verkey') if isinstance(did_info, dict) else None

# Hacer público el DID en el wallet tenant
sc, pub = post(f"{ADMIN_URL}/wallet/did/public", json_payload={'did': issuer_did}, headers=issuer_headers)
print('issuer set public', sc, pub)

# Registrar NYM en ledger (usa la admin del agente que tiene permisos trustee/steward)
sc, reg = post(f"{ADMIN_URL}/ledger/register-nym", json_payload={'did': issuer_did, 'verkey': issuer_verkey, 'alias': 'IssuerTest', 'role': 'ENDORSER'}, headers=issuer_headers)
print('register nym issuer', sc, reg)

# Holder DID
sc, did_info_h = post(f"{ADMIN_URL}/wallet/did/create", json_payload={'method': 'sov', 'options': {'key_type': 'ed25519'}}, headers=holder_headers)
holder_did = did_info_h.get('result', {}).get('did')
holder_verkey = did_info_h.get('result', {}).get('verkey')
sc, pub2 = post(f"{ADMIN_URL}/wallet/did/public", json_payload={'did': holder_did}, headers=holder_headers)
sc, reg2 = post(f"{ADMIN_URL}/ledger/register-nym", json_payload={'did': holder_did, 'verkey': holder_verkey, 'alias': 'HolderTest', 'role': 'ENDORSER'}, headers=holder_headers)
print('holder did register', sc, reg2)

print('\nissuer_did=', issuer_did)
print('holder_did=', holder_did)


In [ ]:
# Registrar Schema y Credential Definition en ledger (Issuer)
schema_payload = {
    'schema_name': 'eVTOL_Credential',
    'schema_version': '1.0',
    'attributes': ['id_puerto', 'state', 'version', 'name']
}
sc, schema_resp = post(f"{ADMIN_URL}/schemas", json_payload=schema_payload, headers=issuer_headers)
print('schema_resp', sc, json.dumps(schema_resp, indent=2))
schema_id = None
if isinstance(schema_resp, dict):
    schema_id = schema_resp.get('schema_id') or schema_resp.get('id') or (schema_resp.get('sent') and schema_resp['sent'].get('schema_id'))
    # ACA-Py sometimes returns under 'schema' or 'sent'
    if not schema_id:
        # fallback: check top-level keys
        schema_id = schema_resp.get('id')
print('schema_id=', schema_id)

# Crear cred def
creddef_payload = {'schema_id': schema_id, 'support_revocation': False, 'tag': 'evtol_tag'}
sc, creddef_resp = post(f"{ADMIN_URL}/credential-definitions", json_payload=creddef_payload, headers=issuer_headers)
print('creddef_resp', sc, json.dumps(creddef_resp, indent=2))
cred_def_id = None
if isinstance(creddef_resp, dict):
    cred_def_id = creddef_resp.get('credential_definition_id') or (creddef_resp.get('sent') and creddef_resp['sent'].get('credential_definition_id'))
print('cred_def_id=', cred_def_id)
assert cred_def_id, 'No se obtuvo credential_definition_id'


In [ ]:
# Issuer crea una invitation y Holder la acepta
sc, inv = post(f"{ADMIN_URL}/connections/create-invitation", headers=issuer_headers)
print('inv create', sc, json.dumps(inv, indent=2))
invitation = inv.get('invitation') if isinstance(inv, dict) else None
inv_url = inv.get('invitation_url') if isinstance(inv, dict) else None

# Holder recibe la invitación
sc, rec = post(f"{ADMIN_URL}/connections/receive-invitation", json_payload=invitation, headers=holder_headers)
print('holder received', sc, json.dumps(rec, indent=2))
# Identificar connection ids
issuer_conn_id = inv.get('connection_id')
holder_conn_id = rec.get('connection_id')
print('issuer_conn_id=', issuer_conn_id)
print('holder_conn_id=', holder_conn_id)

# Esperar a que la conexión pase a active (polling)
ok, info = wait_for_connection(ADMIN_URL, issuer_conn_id, headers=issuer_headers, timeout=60)
print('connection active?', ok)
print(json.dumps(info, indent=2))


In [ ]:
# Enviar oferta de credencial (Issuer -> Holder). Asegúrate connection state == active
credential_preview = [
    {'name':'id_puerto','value':'puerto_42'},
    {'name':'state','value':'ACTIVE'},
    {'name':'version','value':'v1'},
    {'name':'name','value':'EVTOL-Alpha'}
]
offer_payload = {
    'connection_id': issuer_conn_id,
    'cred_def_id': cred_def_id,
    'credential_preview': {
        '@type': 'issue-credential/1.0/credential-preview',
        'attributes': credential_preview
    },
    'auto_issue': True,
    'auto_remove': False
}
sc, resp_offer = post(f"{ADMIN_URL}/issue-credential/send-offer", json_payload=offer_payload, headers=issuer_headers)
print('offer status', sc)
print(json.dumps(resp_offer, indent=2))
print('Ahora en el holder revisa /issue-credential/records y /credentials')


In [ ]:
# Holder: comprobar records e intentar almacenar la credencial cuando esté en request_received/state 'request_received' or 'credential_issued'
import time
def find_offer(records):
    for r in records:
        if r.get('cred_def_id') == cred_def_id:
            return r
    return None

for _ in range(30):
    sc, records = get(f"{ADMIN_URL}/issue-credential/records", headers=holder_headers)
    if isinstance(records, dict) and 'results' in records:
        records_list = records['results']
    else:
        records_list = records if isinstance(records, list) else []
    found = find_offer(records_list)
    print('found:', found and found.get('state'))
    if found and found.get('state') in ['request_received','credential_issued','done']:
        print('offer state now:', found.get('state'))
        cred_ex_id = found.get('cred_ex_id')
        if found.get('state') == 'credential_issued':
            sc_store, res_store = post(f"{ADMIN_URL}/issue-credential/records/{cred_ex_id}/store", headers=holder_headers)
            print('store status', sc_store, res_store)
        break
    time.sleep(2)

# Finalmente listar credenciales guardadas
sc, creds = get(f"{ADMIN_URL}/credentials", headers=holder_headers)
print('stored creds status', sc)
print(json.dumps(creds, indent=2))


### Fin del notebook de flujo de emisión

- Si todo salió bien, verás en la celda anterior la credencial almacenada en el Holder (`/credentials`).
- Si algo falla, copia la salida de error y la revisamos. Comúnmente los problemas son: conexión no activa, cred_def_id inexistente o permisos en el ledger.

**Notas de seguridad:** este notebook usa `admin-insecure-mode` (sin auth) para simplificar pruebas locales. No use este modo en producción.
